In [ ]:
from machine import Pin
import time
import stepper_motor

robarm_motor = step_motor(0,1,2,3)
robarm_motor.set_PWM(20)

bumper_ground_in = Pin(12, Pin.IN, Pin.PULL_DOWN)
bumper_ground_out = Pin(13, Pin.OUT)

bumper_truckbed_in = Pin(14, Pin.IN, Pin.PULL_DOWN)
bumper_truckbed_out = Pin(15, Pin.OUT)

elektromagnet_pin = Pin(16, Pin.OUT)

class robarm_control:
    def __init__(self, robarm_motor, bumper_ground_in, bumper_ground_out, bumper_truckbed_in, bumper_truckbed_out):
        self.motor = robarm_motor
        self.bumper_ground_in = bumper_ground_in
        self.bumper_ground_out = bumper_ground_out
        self.bumper_truckbed_in = bumper_truckbed_in
        self.bumper_truckbed_out = bumper_truckbed_out

    
    def bumper_ground_check(self):
        return self.bumper_ground_in.value()

    def bumper_truckbed_check(self):
        return self.bumper_truckbed_in.value()
    
    def move_up_to_truckbed(self):
        self.motor.set_direction(-1)
        while not self.bumper_truckbed_check():
            self.motor.step()
            time.sleep(0.01)
        self.motor.release()

    def move_down_to_ground(self):
        self.motor.set_direction(1)
        while not self.bumper_ground_check():
            self.motor.step()
            time.sleep(0.01)
        self.motor.release()
    


class electromagnet_control:    
    def __init__(self, pin):
        self.pin = pin

    def turn_on(self): 
        self.pin.value(1)

    def turn_off(self):
        self.pin.value(0)


robarm = robarm_control(robarm_motor, bumper_ground_in, bumper_ground_out, bumper_truckbed_in, bumper_truckbed_out)
electromagnet = electromagnet_control(elektromagnet_pin)
robarm_motor.set_microsteps(12)

while True: 
    robarm.move_down_to_ground()

    if bumper_ground_in.value() == 1:
        electromagnet.turn_on()
        time.sleep(1)
        robarm.move_up_to_truckbed()
        time.sleep(1)
        electromagnet.turn_off()
        time.sleep(1)
        





# Insert the class for your final version of the Stepper Motor 
from machine import PWM
from math import cos, pi

class step_motor:
    def __init__(self, pin1, pin2, pin3, pin4):
        '''
        Motor Pin 1-4, 
        '''
        # Initialize PWM on motor control pins
        self.pwm1 = PWM(pin1, freq=18000)
        self.pwm2 = PWM(pin2, freq=18000)
        self.pwm3 = PWM(pin3, freq=18000)
        self.pwm4 = PWM(pin4, freq=18000)

        # Max duty cycle for 16-bit PWM
        self.duty = 65535  

        # Define the step sequence for a 4-step motor (full step)
        # Just to have a starting point if no microstepping is set
        self.step_sequence = [
            [int(self.duty*0.2), 0, 0, 0],
            [0, int(self.duty*0.2), 0, 0],
            [0, 0, int(self.duty*0.2), 0],
            [0, 0, 0, int(self.duty*0.2)]
        ]
        # Current position in the step sequence
        self.current_step = 0
        self.current_microstep = 0
        self.direction = 1 # forwards = 1 and backwards = -1
        self.microsteps = 1 # default to full step

    def release(self):
        '''releases motor "to save power and prevent overheating"'''
        self.pwm1.duty_u16(0)
        self.pwm2.duty_u16(0)
        self.pwm3.duty_u16(0)
        self.pwm4.duty_u16(0)

    def set_direction(self, direction):
        '''
        sets the direction of the motor\n
        direction = 1 for forwards, -1 for backwards 
        '''
        # validate input
        if direction != 1 and direction != -1:
            raise ValueError("Direction must be either 1 or -1")
        # set direction
        self.direction = direction
        
    def set_PWM(self, PWM):
        '''sets the PWM duty cycle (0-100%)'''
        # validate input
        if PWM < 0 or PWM > 100:
            raise ValueError("PWM must be between 0 and 100")
        # set duty cycle
        self.duty = int(PWM / 100 * 65535)
        self.set_microsteps(self.microsteps)

    def set_microsteps(self, microsteps, PWM=None):
        ''' gernates a new step sequence for the given number of microsteps'''
        # validate input
        # must be a positive integer
        if PWM is not None:
            # validate input
            if PWM < 0 or PWM > 100:
                raise ValueError("PWM must be between 0 and 100")
            # set duty cycle
            self.duty = int(PWM / 100 * 65535)

        if microsteps < 1:
            raise ValueError("Microsteps must be at least 1")
        
        self.microsteps = microsteps
        # 1 = full step, 2 = half step, 4 = quarter step, 8 = eighth step etc'''

        # gernate new step sequence for microstepping
        # clear existing sequence
        self.step_sequence = []
        # generate new sequence (microsteps per full step * number of full steps (4))
        for i in range(microsteps*4):
            # calculate the sine wave values for each coil
            # angle ranges from 0 to 2*pi over the full step sequence
            angle = (i / microsteps) * (pi / 2)  # Calculate the angle for the sine wave

            # calculate the PWM duty cycle for each coil using a sine wave
            # cos(angle - phase shift) to get the correct phase for each coil
            # max(0, ...) to ensure no negative duty cycles
            step = [
                max(0,int(round((cos(angle - 0 * (pi / 2)) * self.duty)))),
                max(0,int(round((cos(angle - 1 * (pi / 2)) * self.duty)))),
                max(0,int(round((cos(angle - 2 * (pi / 2)) * self.duty)))),
                max(0,int(round((cos(angle - 3 * (pi / 2)) * self.duty))))
            ]
            # append the calculated step to the step sequence
            self.step_sequence.append(step)

    def step(self):
        '''step the motor in th'''
        # get the current step from the sequence
        step = self.step_sequence[self.current_step]
        # set the PWM duty cycle for each coil
        self.pwm1.duty_u16(step[0])
        self.pwm2.duty_u16(step[1])
        self.pwm3.duty_u16(step[2])
        self.pwm4.duty_u16(step[3])
        # update the current step based on the direction
        if self.direction == -1:
            self.current_step = (self.current_step - 1) % len(self.step_sequence)
        else:
            self.current_step = (self.current_step + 1) % len(self.step_sequence)


